In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

In [2]:
import os
import pandas as pd
import polars as pl
import numpy as np
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, ExtraTreesRegressor
import kaggle_evaluation.default_inference_server

global_rf_model = None
global_hgb_model = None
global_et_model = None

def predict(test: pl.DataFrame) -> float:
    global global_rf_model, global_hgb_model, global_et_model
    
    features = [
        'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 
        'P1', 'P2', 'P3', 'P4', 'P5', 
        'V1', 'V2', 'V3', 'V4', 
        'E1', 'E2', 'I1', 'I2', 'S1'
    ]
    target = 'market_forward_excess_returns'

    if global_rf_model is None:
        train_df = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
        train_df = train_df.dropna(subset=[target])
        train_df[features] = train_df[features].fillna(0)
        
        rf = RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
        rf.fit(train_df[features], train_df[target])
        global_rf_model = rf
        
        et = ExtraTreesRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
        et.fit(train_df[features], train_df[target])
        global_et_model = et
        
        hgb = HistGradientBoostingRegressor(max_iter=100, max_depth=5, learning_rate=0.05, random_state=42)
        hgb.fit(train_df[features], train_df[target])
        global_hgb_model = hgb
        
    test_pd = test.to_pandas()
    
    X_test = pd.DataFrame(index=test_pd.index)
    for f in features:
        if f in test_pd.columns:
            X_test[f] = test_pd[f]
        else:
            X_test[f] = 0.0
            
    X_test = X_test.fillna(0)
    
    pred_rf = global_rf_model.predict(X_test[features])[0]
    pred_et = global_et_model.predict(X_test[features])[0]
    pred_hgb = global_hgb_model.predict(X_test[features])[0]
    
    final_pred = 0.25 * pred_rf + 0.25 * pred_et + 0.5 * pred_hgb
    
    current_volatility = X_test['V1'].values[0] if 'V1' in X_test.columns else 0
    
    allocation = final_pred * 300.0
    
    if allocation < 0:
        allocation = 0.0
        
    if current_volatility > 50:
        allocation = allocation * 0.5
        
    return float(max(0.0, min(2.0, allocation)))

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    print("Running local gateway...")
    inference_server.run_local_gateway(
        ('/kaggle/input/hull-tactical-market-prediction/',) 
    )

Running local gateway...
